# PySpark: Distributed Data Processing at Scale

## What Is PySpark?

Imagine you have 1 billion rows of data — it won't fit on any single computer's memory.  
**PySpark** lets you spread that data across 100 machines and process it in parallel.  
Each machine handles a chunk; Spark coordinates them all as if it were one giant computer.

**PySpark** is the Python API for **Apache Spark** — an open-source distributed computing engine.  
Originally from UC Berkeley's AMPLab (2009), now the most popular big data processing framework.

Key capabilities:
- **Scale**: process petabytes of data across thousands of machines
- **Speed**: in-memory processing (100× faster than Hadoop MapReduce)
- **Unified**: batch, streaming, ML (MLlib), and SQL in one framework
- **Lazy evaluation**: builds execution plan, optimizes, then runs
- **Fault tolerant**: if a machine fails, Spark recomputes that chunk automatically

## Resources

- **Docs**: [https://spark.apache.org/docs/latest/api/python/](https://spark.apache.org/docs/latest/api/python/)
- **GitHub**: [https://github.com/apache/spark](https://github.com/apache/spark)
- **YouTube — PySpark Tutorial**: [https://www.youtube.com/watch?v=_C8kWso4ne4](https://www.youtube.com/watch?v=_C8kWso4ne4)
- **Databricks Free Community Edition**: [https://community.cloud.databricks.com/](https://community.cloud.databricks.com/)

## Installation

```bash
pip install pyspark
# PySpark includes a local Spark instance — no cluster needed for development!

# Java is required (PySpark runs on the JVM)
# Install Java 11:
#   macOS:   brew install openjdk@11
#   Ubuntu:  sudo apt install openjdk-11-jdk
#   Windows: download from https://adoptium.net/
```

In [ ]:
import numpy as np
import os, time, tempfile, shutil

try:
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    from pyspark.sql.types import (
        StructType, StructField, StringType, DoubleType, IntegerType
    )
    SPARK_AVAILABLE = True
    import pyspark
    print(f"PySpark version: {pyspark.__version__}")
except ImportError:
    SPARK_AVAILABLE = False
    print("PySpark not installed — simulated output shown.")
    print("Install: pip install pyspark  (requires Java 11+)")

# Create synthetic sales data
np.random.seed(42)
N = 100_000

regions    = np.random.choice(['North', 'South', 'East', 'West'], N).tolist()
categories = np.random.choice(['Electronics', 'Clothing', 'Food', 'Books'], N).tolist()
sales      = np.round(np.random.exponential(200, N), 2).tolist()
quantities = np.random.randint(1, 50, N).tolist()

print(f"\nSynthetic data ready: {N:,} rows")

## Core Concept 1: SparkSession — The Entry Point

**SparkSession** is the gateway to all Spark functionality.  
In local mode (your laptop), Spark uses all available CPU cores as workers.

In [ ]:
if SPARK_AVAILABLE:
    # Create a local SparkSession
    # 'local[*]' means use all available CPU cores locally
    # In production: 'spark://master:7077' for a real cluster
    spark = (
        SparkSession.builder
        .appName("SalesAnalysis")           # name shown in Spark UI
        .master("local[*]")                 # local: use all cores
        .config("spark.sql.shuffle.partitions", "4")  # default 200, too high for local
        .config("spark.driver.memory", "2g")
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("ERROR")  # reduce verbose logging

    print(f"Spark version: {spark.version}")
    print(f"Master:        {spark.sparkContext.master}")
    print(f"App name:      {spark.sparkContext.appName}")
    print(f"Parallelism:   {spark.sparkContext.defaultParallelism} cores")

    # Create DataFrame from Python lists
    rows = list(zip(regions, categories, sales, quantities))
    schema = StructType([
        StructField('region',   StringType(),  nullable=False),
        StructField('category', StringType(),  nullable=False),
        StructField('sales',    DoubleType(),  nullable=False),
        StructField('quantity', IntegerType(), nullable=False),
    ])
    df = spark.createDataFrame(rows, schema=schema)

    print(f"\nDataFrame created:")
    print(f"  Partitions: {df.rdd.getNumPartitions()}")
    df.printSchema()
    df.show(5)

else:
    print("SparkSession (simulated):")
    print()
    print("  spark = (")
    print("      SparkSession.builder")
    print("      .appName('SalesAnalysis')")
    print("      .master('local[*]')         # use all cores locally")
    print("      # In production: .master('spark://master:7077')")
    print("      .getOrCreate()")
    print("  )")
    print()
    print("  Spark version: 3.5.0")
    print("  Master:        local[8]  (8 cores)")
    print()
    print("  schema:")
    print("    root")
    print("    |-- region:   string")
    print("    |-- category: string")
    print("    |-- sales:    double")
    print("    |-- quantity: integer")
    print()
    print("  +--------+-----------+------+--------+")
    print("  | region | category  | sales|quantity|")
    print("  +--------+-----------+------+--------+")
    print("  | North  |Electronics|182.34|      12|")
    print("  | South  |Clothing   | 45.67|       3|")
    print("  +--------+-----------+------+--------+")

## Core Concept 2: DataFrame Transformations

Spark DataFrames are **lazy** — transformations build a plan, **actions** trigger execution.

| Transformations (lazy) | Actions (execute) |
|------------------------|-------------------|
| `select()`, `filter()`, `groupBy()` | `show()`, `count()`, `collect()` |
| `withColumn()`, `join()`, `orderBy()` | `write.csv()`, `toPandas()` |
| `distinct()`, `drop()` | `first()`, `take(n)` |

In [ ]:
if SPARK_AVAILABLE:
    # SELECT: choose columns
    print("1. Select columns:")
    df.select('region', 'sales').show(3)

    # FILTER: like SQL WHERE
    print("2. Filter (sales > 500 AND region == 'North'):")
    high_north = df.filter((F.col('sales') > 500) & (F.col('region') == 'North'))
    print(f"   Count: {high_north.count():,}")
    high_north.show(3)

    # WITHCOLUMN: add a new column
    print("3. Add computed columns with withColumn:")
    df2 = (
        df
        .withColumn('revenue', F.col('sales') * F.col('quantity'))
        .withColumn('log_sales', F.log10(F.col('sales')))
        .withColumn('region_upper', F.upper(F.col('region')))
    )
    df2.show(3)

    # DISTINCT: unique values
    print("4. Distinct regions:")
    df.select('region').distinct().show()

else:
    print("DataFrame transformations (simulated):")
    print()
    print("  # Select columns")
    print("  df.select('region', 'sales').show(3)")
    print()
    print("  # Filter (lazy — builds plan, doesn't execute)")
    print("  df.filter((F.col('sales') > 500) & (F.col('region') == 'North'))")
    print("  # .count() or .show() triggers execution")
    print()
    print("  # Add columns")
    print("  df.withColumn('revenue', F.col('sales') * F.col('quantity'))")
    print("    .withColumn('log_sales', F.log10(F.col('sales')))")
    print()
    print("  # Distinct")
    print("  df.select('region').distinct().show()")
    print("  +--------+")
    print("  | region |")
    print("  +--------+")
    print("  |  North |")
    print("  |  South |")
    print("  |   East |")
    print("  |   West |")
    print("  +--------+")

## Core Concept 3: GroupBy and Aggregations

In [ ]:
if SPARK_AVAILABLE:
    print("GroupBy with multiple aggregations:")
    summary = (
        df
        .withColumn('revenue', F.col('sales') * F.col('quantity'))
        .groupBy('region', 'category')
        .agg(
            F.sum('revenue').alias('total_revenue'),
            F.mean('sales').alias('avg_sales'),
            F.max('sales').alias('max_sales'),
            F.sum('quantity').alias('total_qty'),
            F.count('*').alias('num_transactions'),
        )
        .orderBy(F.desc('total_revenue'))
    )
    summary.show(10, truncate=False)

    print("DataFrame statistics:")
    df.select('sales', 'quantity').describe().show()

else:
    print("GroupBy aggregation (simulated):")
    print()
    print("  df.withColumn('revenue', F.col('sales') * F.col('quantity'))")
    print("    .groupBy('region', 'category')")
    print("    .agg(")
    print("        F.sum('revenue').alias('total_revenue'),")
    print("        F.mean('sales').alias('avg_sales'),")
    print("        F.count('*').alias('num_transactions'),")
    print("    )")
    print("    .orderBy(F.desc('total_revenue'))")
    print("    .show(10)")
    print()
    print("  +------+-----------+-------------+---------+")
    print("  |region|category   |total_revenue|avg_sales|")
    print("  +------+-----------+-------------+---------+")
    print("  |West  |Electronics|  363,456.78 |  198.34 |")
    print("  |North |Electronics|  360,123.45 |  197.82 |")
    print("  +------+-----------+-------------+---------+")

## Core Concept 4: Spark SQL

You can register a DataFrame as a **SQL table** and write plain SQL queries.  
Same execution engine — SQL and DataFrame API produce identical results.

In [ ]:
if SPARK_AVAILABLE:
    # Register DataFrame as a temporary SQL view
    df.withColumn('revenue', F.col('sales') * F.col('quantity')) \
      .createOrReplaceTempView('sales')

    print("Spark SQL — same results as DataFrame API:")
    sql_result = spark.sql("""
        SELECT
            region,
            category,
            SUM(revenue)      AS total_revenue,
            AVG(sales)        AS avg_sales,
            COUNT(*)          AS num_transactions,
            RANK() OVER (
                PARTITION BY region
                ORDER BY SUM(revenue) DESC
            )                 AS rank_in_region
        FROM sales
        GROUP BY region, category
        HAVING SUM(revenue) > 100000
        ORDER BY region, rank_in_region
        LIMIT 20
    """)
    sql_result.show(truncate=False)

    # List registered tables
    print("Registered tables:")
    spark.catalog.listTables()

else:
    print("Spark SQL (simulated):")
    print()
    print("  # Register as SQL table")
    print("  df.createOrReplaceTempView('sales')")
    print()
    print("  # Run SQL — same execution engine as DataFrame API")
    print("  spark.sql('''")
    print("      SELECT")
    print("          region, category,")
    print("          SUM(revenue) AS total_revenue,")
    print("          RANK() OVER (PARTITION BY region ORDER BY SUM(revenue) DESC)")
    print("      FROM sales")
    print("      GROUP BY region, category")
    print("      ORDER BY region, rank_in_region")
    print("  ''').show()")
    print()
    print("  Spark SQL supports: window functions, CTEs, subqueries, UDFs, ...")

## Core Concept 5: Reading and Writing Data

Spark reads from many sources: CSV, Parquet, JSON, Delta Lake, JDBC (databases), Hive, S3, HDFS.

In [ ]:
if SPARK_AVAILABLE:
    tmp_dir = tempfile.mkdtemp()

    # Write to Parquet (most common format for Spark)
    parquet_path = os.path.join(tmp_dir, 'sales_parquet')
    df.write.mode('overwrite').parquet(parquet_path)
    print(f"Written Parquet to: {parquet_path}")
    print(f"Files: {os.listdir(parquet_path)}")

    # Write to CSV
    csv_path = os.path.join(tmp_dir, 'sales_csv')
    df.write.mode('overwrite').option('header', 'true').csv(csv_path)
    print(f"\nWritten CSV to: {csv_path}")
    print()

    # Read back
    df_parquet = spark.read.parquet(parquet_path)
    df_csv     = spark.read.option('header', 'true').option('inferSchema', 'true').csv(csv_path)

    print(f"Read Parquet: {df_parquet.count():,} rows")
    print(f"Read CSV:     {df_csv.count():,} rows")
    print()

    # Partitioned write (important for query performance)
    partitioned_path = os.path.join(tmp_dir, 'sales_partitioned')
    df.write.mode('overwrite').partitionBy('region').parquet(partitioned_path)
    print(f"Partitioned by region:")
    for d in sorted(os.listdir(partitioned_path)):
        if d.startswith('region='):
            print(f"  {d}/")

    shutil.rmtree(tmp_dir)

else:
    print("File I/O (simulated):")
    print()
    print("  # Write Parquet (recommended for big data)")
    print("  df.write.mode('overwrite').parquet('s3://bucket/sales/')")
    print()
    print("  # Write CSV")
    print("  df.write.mode('overwrite').option('header', 'true').csv('s3://bucket/sales_csv/')")
    print()
    print("  # Read")
    print("  spark.read.parquet('s3://bucket/sales/')")
    print("  spark.read.option('header','true').option('inferSchema','true').csv('data.csv')")
    print()
    print("  # Partitioned write (creates directories: region=North/, region=South/, ...)")
    print("  # Queries filtering by region only read that partition — huge speedup")
    print("  df.write.partitionBy('region').parquet('s3://bucket/sales_partitioned/')")
    print()
    print("  Directory structure:")
    print("    sales_partitioned/")
    print("      region=East/   part-00000.parquet")
    print("      region=North/  part-00001.parquet")
    print("      region=South/  part-00002.parquet")
    print("      region=West/   part-00003.parquet")

## Core Concept 6: Joins in Spark

Spark supports all SQL join types. For big data, join strategy matters:
- **Broadcast join**: small table broadcast to all workers — no shuffle (fast)
- **Sort-merge join**: default — both sides sorted and merged
- **Shuffle hash join**: one side hashed, other side hashes matched

In [ ]:
if SPARK_AVAILABLE:
    # Create a small lookup table (products)
    products_data = [
        ('Electronics', 'High-value tech products', 'premium'),
        ('Clothing',    'Fashion and apparel',      'standard'),
        ('Food',        'Grocery and perishables',  'standard'),
        ('Books',       'Physical and digital',     'low-margin'),
    ]
    products = spark.createDataFrame(
        products_data,
        schema=['category', 'description', 'tier']
    )

    print("Products lookup table:")
    products.show()

    # Broadcast join: products is small → broadcast to all executors
    # No shuffle needed — 10-100× faster than sort-merge join
    from pyspark.sql.functions import broadcast

    enriched = df.join(
        broadcast(products),  # hint to Spark: broadcast this small table
        on='category',
        how='inner'
    )

    print("Enriched DataFrame (with broadcast join):")
    enriched.show(5)
    print(f"Enriched count: {enriched.count():,}")

    # Left join example
    left_joined = df.join(products, on='category', how='left')
    print(f"Left join count: {left_joined.count():,}")

else:
    print("Joins in Spark (simulated):")
    print()
    print("  # Small table → broadcast it (no shuffle, very fast)")
    print("  from pyspark.sql.functions import broadcast")
    print("  enriched = df.join(")
    print("      broadcast(products),  # small lookup table")
    print("      on='category',")
    print("      how='inner'")
    print("  )")
    print()
    print("  Join types: 'inner', 'left', 'right', 'outer', 'semi', 'anti', 'cross'")
    print()
    print("  Join strategies (Spark chooses automatically):")
    print("    Broadcast:  small table (<10MB) broadcast to all workers")
    print("    Sort-merge: both tables sorted then merged (default for large)")
    print("    Force broadcast: broadcast(df) hint")

## Core Concept 7: User-Defined Functions (UDFs)

UDFs let you apply custom Python logic to each row.  
**Important**: UDFs are slow (Python ↔ JVM overhead). Use built-in Spark functions when possible.

In [ ]:
if SPARK_AVAILABLE:
    from pyspark.sql.types import StringType

    # Define a Python function
    def sales_tier(sales_val):
        """Categorize sales amount into tiers."""
        if sales_val < 50:    return 'Low'
        elif sales_val < 200: return 'Medium'
        elif sales_val < 500: return 'High'
        else:                 return 'Premium'

    # Register as Spark UDF
    sales_tier_udf = F.udf(sales_tier, StringType())

    df_with_tier = df.withColumn('tier', sales_tier_udf(F.col('sales')))
    print("UDF applied — sales tier:")
    df_with_tier.show(5)

    # Better: use Spark's built-in when/otherwise (much faster, stays in JVM)
    df_with_tier_native = df.withColumn('tier',
        F.when(F.col('sales') < 50,   'Low')
         .when(F.col('sales') < 200,  'Medium')
         .when(F.col('sales') < 500,  'High')
         .otherwise('Premium')
    )
    print("Same result with native when/otherwise (much faster than UDF):")
    df_with_tier_native.groupBy('tier').count().orderBy('tier').show()

else:
    print("UDFs (simulated):")
    print()
    print("  # Python UDF (slow — crosses JVM/Python boundary per row)")
    print("  def sales_tier(val):")
    print("      if val < 50:    return 'Low'")
    print("      elif val < 200: return 'Medium'")
    print("      else:           return 'High'")
    print()
    print("  sales_tier_udf = F.udf(sales_tier, StringType())")
    print("  df.withColumn('tier', sales_tier_udf(F.col('sales')))")
    print()
    print("  # Better: native Spark functions (stay in JVM, 10-100× faster)")
    print("  df.withColumn('tier',")
    print("      F.when(F.col('sales') < 50,   'Low')")
    print("       .when(F.col('sales') < 200,  'Medium')")
    print("       .otherwise('High')")
    print("  )")

## Core Concept 8: Partitioning and Performance

**Partitions** are chunks of data distributed across workers.  
Too few → workers idle; too many → coordination overhead.

In [ ]:
if SPARK_AVAILABLE:
    print("Partitioning:")
    print(f"  Current partitions: {df.rdd.getNumPartitions()}")

    # Repartition: redistribute data into N partitions (with shuffle)
    df_4 = df.repartition(4)
    print(f"  After repartition(4): {df_4.rdd.getNumPartitions()}")

    # Coalesce: reduce partitions (no shuffle, faster than repartition)
    df_2 = df_4.coalesce(2)
    print(f"  After coalesce(2):    {df_2.rdd.getNumPartitions()}")

    # Repartition by column (good for joins and groupBy on that column)
    df_region = df.repartition(4, 'region')
    print(f"  After repartition(4, 'region'): {df_region.rdd.getNumPartitions()}")
    print("  All North rows → same partition → no shuffle needed for groupBy('region')")

    # Cache/persist: keep in memory to avoid recomputation
    df.cache()  # equivalent to df.persist()
    print(f"\nCache: df.cache() — marks for in-memory storage")
    df.count()  # triggers caching
    print(f"  Now cached — subsequent queries on df skip reread")
    df.unpersist()  # release cache when done

else:
    print("Partitioning and caching (simulated):")
    print()
    print("  df.rdd.getNumPartitions()   # current partition count")
    print()
    print("  # Repartition: shuffle data into N partitions")
    print("  df.repartition(200)                # good default for large data")
    print("  df.repartition(4, 'region')        # co-locate by column value")
    print()
    print("  # Coalesce: reduce partitions without shuffle (only merge)")
    print("  df.coalesce(1)              # combine to 1 file (for writing)")
    print()
    print("  # Cache: store in memory to avoid recomputation")
    print("  df.cache()                  # cache after first .count()/.show()")
    print("  df.persist(StorageLevel.MEMORY_AND_DISK)  # spill to disk if OOM")
    print("  df.unpersist()              # release when done")
    print()
    print("  Rule of thumb:")
    print("    partitions = max(2 × num_cores, data_size_GB × 100)")
    print("    target: 100-500 MB per partition")

## Common Pitfalls

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| Calling `.collect()` on huge DataFrames | OutOfMemoryError on driver | Use `.show()`, `.take(n)`, or write to file |
| Python UDFs for everything | Very slow | Use `F.when()`, `F.regexp_replace()`, etc. (native functions) |
| Too many or too few partitions | Slow jobs | Target 100-500MB per partition; use `repartition()` |
| Forgetting to cache() iterative DFs | Recomputed from scratch each use | `.cache()` before multiple uses of the same DF |
| Reading CSV with default settings | Slow, wrong types | Use `inferSchema=True` or explicit schema; prefer Parquet |
| Shuffles everywhere (joins, groupBy) | Slow network-bound jobs | Use `broadcast()` for small tables, partition by join key |
| `spark.sql.shuffle.partitions=200` | Too many small tasks locally | Set to 4-8 for local mode |

## Mini Project: End-to-End Sales Analysis

In [ ]:
if SPARK_AVAILABLE:
    print("=" * 60)
    print("END-TO-END SALES ANALYSIS WITH PYSPARK")
    print("=" * 60)
    print()

    t0 = time.time()

    # Full pipeline
    result = (
        df
        # Step 1: Add computed columns
        .withColumn('revenue', F.round(F.col('sales') * F.col('quantity'), 2))
        .withColumn('tier',
            F.when(F.col('sales') < 50,   'Low')
             .when(F.col('sales') < 200,  'Medium')
             .when(F.col('sales') < 500,  'High')
             .otherwise('Premium')
        )

        # Step 2: Register for SQL
        .createOrReplaceTempView('enriched_sales') or

        # Step 3: SQL query with window function
        spark.sql("""
            SELECT
                region,
                category,
                tier,
                ROUND(SUM(revenue), 2)    AS total_revenue,
                ROUND(AVG(sales), 2)      AS avg_sale,
                COUNT(*)                  AS num_orders,
                RANK() OVER (
                    PARTITION BY region
                    ORDER BY SUM(revenue) DESC
                )                         AS rank_in_region
            FROM enriched_sales
            GROUP BY region, category, tier
            ORDER BY region, rank_in_region
        """)
    )

    result.show(20, truncate=False)
    print(f"Computed in {(time.time()-t0)*1000:.0f} ms for {N:,} input rows")

    # Clean up
    spark.stop()

else:
    print("Mini Project Output (simulated):")
    print()
    print("  +------+-----------+-------+--------------+--------+-----------+")
    print("  |region|category   |tier   |total_revenue |avg_sale|rank_in_reg|")
    print("  +------+-----------+-------+--------------+--------+-----------+")
    print("  |East  |Electronics|Medium |  363,456.78  |198.34  |          1|")
    print("  |East  |Sports     |Medium |  349,123.45  |195.23  |          2|")
    print("  |East  |Food       |Medium |  340,234.56  |193.45  |          3|")
    print("  |East  |Clothing   |Low    |   12,345.67  | 45.67  |          4|")
    print("  +------+-----------+-------+--------------+--------+-----------+")
    print("  ...")
    print()
    print("  Computed in 2,340 ms for 100,000 input rows")
    print("  (Spark has JVM startup overhead; faster for 100M+ rows)")

## Interview Questions and Answers

In [ ]:
qa = [
    {"q": "What is lazy evaluation in Spark and why does it matter?",
     "a": """Lazy evaluation = transformations build a DAG (Directed Acyclic Graph);
actions trigger optimization + execution of the entire DAG.

Transformations (lazy): select(), filter(), groupBy(), join(), withColumn()
Actions (execute DAG):  show(), count(), collect(), write(), toPandas()

Why it matters:

1. Catalyst Optimizer:
   Spark rewrites your query plan automatically:
   - Predicate pushdown: apply filter before join (fewer rows to join)
   - Projection pruning: only read needed columns
   - Constant folding: precompute constants

2. Tungsten engine:
   Physical optimizer generates optimized JVM bytecode.
   Uses off-heap memory, columnar processing, vectorized execution.

3. Single pass through data:
   Chain of .filter().select().withColumn() runs in one pass,
   not three separate scans of the data.

Example:
  df.filter(...).select(...).groupBy(...).count()
  # Spark optimizes: push filter before select, compute count in parallel"""},

    {"q": "What is a shuffle and why is it expensive?",
     "a": """A shuffle is when Spark needs to redistribute data across executors
so that rows that belong together end up on the same machine.

When shuffles happen:
  - groupBy('col'): all rows with same col value must go to same executor
  - join(other, on='key'): matching keys must be on same executor
  - orderBy(): total sort requires all data to be redistributed
  - distinct(): deduplicate across all nodes

Why expensive:
  1. Network I/O: data travels between machines over the network
  2. Disk I/O: shuffle data spills to disk if memory insufficient
  3. Serialization: data serialized/deserialized for network transfer

How to minimize:
  - Broadcast join for small tables (no shuffle)
  - Partition by join key before joining (avoid re-shuffle)
  - Reduce shuffle partitions: spark.sql.shuffle.partitions=200
    (default is 200; too many means lots of tiny tasks)
  - Filter aggressively before joining (fewer rows to shuffle)"""},

    {"q": "Spark vs pandas vs Polars — when do you use each?",
     "a": """Choose based on data size, infrastructure, and use case:

pandas:
  - Data: < 5GB (fits in laptop RAM)
  - Team: familiar with pandas syntax
  - Use: exploration, prototyping, small production jobs
  - Pro: simple, rich ecosystem (matplotlib, sklearn, etc.)
  - Con: single-threaded, memory-inefficient

Polars:
  - Data: 1-100GB on a single machine
  - Team: needs pandas speed without a cluster
  - Use: fast ETL, feature engineering, analytics
  - Pro: 5-30× faster than pandas, multi-core, query optimizer
  - Con: smaller ecosystem, not on clusters

PySpark:
  - Data: > 100GB, or on a cluster (Databricks, EMR, etc.)
  - Team: has Spark/Databricks infrastructure
  - Use: enterprise ETL, big data ML, streaming
  - Pro: unlimited scale, MLlib, streaming, Delta Lake
  - Con: JVM overhead (high latency), complex debugging

Rule:
  < 5GB  → pandas
  5-100GB → Polars (same machine)
  > 100GB → PySpark (cluster required)"""},

    {"q": "What is the Spark execution model (Driver, Executors, Tasks)?",
     "a": """Spark has a master-worker architecture:

Driver (1 per application):
  - Your Python script runs here
  - Builds the DAG of transformations
  - Divides work into Tasks and sends to executors
  - Collects results when .collect() is called
  - Only machine that needs to fit final result in memory

Executors (1+ per worker node):
  - JVM processes that do the actual computation
  - Each executor has N cores and some memory
  - Runs Tasks in parallel

Tasks (1 per partition per stage):
  - Smallest unit of work
  - Processes one partition of data
  - 200 partitions × 1 stage = 200 tasks (run in parallel)

Stages: separated by shuffle boundaries
  Stage 1: read + filter + map (no shuffle)
  === Shuffle === (network transfer)
  Stage 2: groupBy + aggregate

Jobs: triggered by one action (e.g. .count())
  One job → multiple stages → many tasks"""},

    {"q": "How would you optimize a slow Spark job?",
     "a": """Optimization checklist:

1. Check Spark UI (port 4040):
   - Find the slowest stage
   - Look for data skew (one partition 10× larger than others)

2. Reduce shuffles:
   - Use broadcast join for small tables (< 10MB)
   - Pre-partition by join/groupBy key

3. Fix data skew:
   - Salting: add random suffix to skewed key, multiply other table
   - Use repartition() to redistribute unevenly distributed data

4. Tune partitions:
   - spark.sql.shuffle.partitions (default 200 — often too many)
   - Target 100-500 MB per partition
   - df.repartition(N) before expensive operations

5. Cache strategically:
   - df.cache() before reusing a DataFrame multiple times
   - Unpersist when done: df.unpersist()

6. Use Parquet instead of CSV:
   - Columnar: Spark reads only needed columns
   - Compressed: less I/O
   - Partition pruning: filter at file level

7. Avoid Python UDFs:
   - Use built-in Spark functions (F.when, F.regexp_replace)
   - If must use Python: use Pandas UDFs (vectorized, 10× faster)"""},

    {"q": "What are RDDs and when would you use them over DataFrames?",
     "a": """RDD (Resilient Distributed Dataset) is Spark's original low-level API.

DataFrame (higher level):
  - Structured: has named columns and dtypes
  - Uses Catalyst optimizer → automatic optimization
  - SQL-like API: .select(), .filter(), .groupBy()
  - Much faster for structured data (optimizer + Tungsten)

RDD (lower level):
  - Unstructured: any Python object
  - No automatic optimization
  - Functional API: .map(), .filter(), .reduce(), .flatMap()
  - More control but more verbose

When to use RDDs:
  - Unstructured data (log parsing, text processing)
  - Need fine-grained control over partitioning
  - Legacy code
  - Complex non-SQL transformations that DataFrames can't express

When to use DataFrames (almost always):
  - Structured/semi-structured data
  - ETL, aggregations, joins
  - ML features (integrates with MLlib)

Rule: start with DataFrame/SQL API.
Use RDD only when DataFrame genuinely can't express what you need."""},
]

for i, item in enumerate(qa, 1):
    print(f"Q{i}: {item['q']}")
    print(f"A:  {item['a'].strip()}")
    print("-" * 65)
    print()

## Summary

| Operation | PySpark API |
|-----------|-------------|
| Start Spark | `SparkSession.builder.appName('name').master('local[*]').getOrCreate()` |
| Create DataFrame | `spark.createDataFrame(rows, schema)` |
| Read Parquet | `spark.read.parquet('path/')` |
| Read CSV | `spark.read.option('header','true').csv('file.csv')` |
| Select columns | `df.select('col1', 'col2')` |
| Filter rows | `df.filter(F.col('x') > 0)` |
| Add column | `df.withColumn('name', expr)` |
| GroupBy | `df.groupBy('col').agg(F.sum('val').alias('total'))` |
| Sort | `df.orderBy(F.desc('col'))` |
| Join | `df.join(other, on='key', how='inner')` |
| Broadcast join | `df.join(broadcast(small), on='key')` |
| SQL query | `spark.sql('SELECT ... FROM view')` |
| Register view | `df.createOrReplaceTempView('name')` |
| Conditional | `F.when(cond, val).otherwise(default)` |
| Window function | `F.rank().over(Window.partitionBy('col').orderBy('val'))` |
| Cache | `df.cache()` / `df.unpersist()` |
| Repartition | `df.repartition(N)` / `df.coalesce(N)` |
| Write Parquet | `df.write.mode('overwrite').parquet('path/')` |
| Stop Spark | `spark.stop()` |

### Next Steps
1. **PySpark getting started**: [https://spark.apache.org/docs/latest/api/python/getting_started/index.html](https://spark.apache.org/docs/latest/api/python/getting_started/index.html)
2. **Databricks free community**: [https://community.cloud.databricks.com/](https://community.cloud.databricks.com/)
3. **MLlib for ML at scale**: [https://spark.apache.org/docs/latest/ml-guide.html](https://spark.apache.org/docs/latest/ml-guide.html)
4. **Next**: AutoML — automated machine learning with AutoGluon, H2O, and Optuna